# Мое решение задачи C

Описание решения:
1) Добавить много фичей. Из представленных данных явно видно, что можно играться с текстовыми фичами. От того, как предобработать текст, будет в основном зависеть скор.
2) В данных есть геоданные, по ним можно создать новые признаки
3) Типично: из одних табличных данных получить другие табличные данные
4) Обучить несколько моделек и взешенно усреднить их предсказания. Практика показала, что catboost справляется гораздо лучше, поэтому у него больший вес. Также обучение по фолдам дает сильный вклад
5) Ну и, конечно, подобрать гиперпараметры, что может дать более сильную позицию на лидерборде

In [ ]:
!unzip reviews.txv.zip
!unzip test.tsv.zip
!unzip train.tsv.zip
!pip install catboost xgboost optuna

## План такой:
1. Загрузить данные (трейн, тест, отзывы)
2. Склеить все отзывы по ID в один большой текст
3. Сгенерировать фичи:
    - Гео-фичи (кластеризация координат)
    - Табличные фичи (разницы, отношения между радиусами 300м/1000м)
    - Демографические фичи (доли, % от общего числа)
    - Текстовые фичи (TF-IDF + SVD) из склеенных отзывов
    - Текстовые фичи (Embeddings E5) из *каждого* отзыва, усредненные по ID, + SVD
4. Собрать всё в один большой датасет
5. Настроить гиперпараметры для CatBoost, XGBoost, LightGBM с помощью Optuna
6. Обучить 3 модели на 5-фолдовой кросс-валидации
7. Сделать взвешенный бленд (ансамбль) из 3-х моделей
8. Сформировать сабмит

In [ ]:
import os
import re
import gc
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import mean_absolute_error

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import optuna
from optuna.samplers import TPESampler

import nltk
from nltk.corpus import stopwords

import torch
from transformers import AutoTokenizer, AutoModel
import random


tqdm.pandas()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_GPU = (DEVICE == "cuda")

CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

E5_MODEL_NAME = "intfloat/multilingual-e5-base"
E5_BATCH = 16 if USE_GPU else 8
E5_MAX_LEN = 256
E5_SVD_DIM = 256
TFIDF_MAX_FEATURES = 15000
TFIDF_SVD_DIM = 200
N_TRIALS = 30
ITERATIONS = 3000

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^а-яa-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def set_seed(seed_value):
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
set_seed(42)

def safe_div(a, b, eps=1e-6):
    return a / (b + eps)

def parse_coords(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.nan, np.nan
    if isinstance(val, (list, tuple)) and len(val) >= 2:
        lon, lat = val[0], val[1]
        return float(lon), float(lat)
    s = str(val)
    nums = re.findall(r'[-+]?\d*\.\d+|[-+]?\d+', s)
    if len(nums) >= 2:
        lon, lat = float(nums[0]), float(nums[1])
        return lon, lat
    return np.nan, np.nan

nltk.download('stopwords', quiet=True)
stop_words = stopwords.words('russian')

# --------------------
# Загрузить данные
# --------------------
train = pd.read_csv("train.tsv", sep='\t')
test = pd.read_csv("test.tsv", sep='\t')
reviews = pd.read_csv("reviews.tsv", sep='\t')

print(f"Train: {train.shape}, Test: {test.shape}, Reviews: {reviews.shape}")

# --------------------
# Склеить все отзывы по ID в один большой текст
# --------------------
reviews_grouped = (
    reviews.groupby('id')['text']
    .progress_apply(lambda x: ' '.join(str(t) for t in x))
    .reset_index()
)
train = train.merge(reviews_grouped, on='id', how='left')
test = test.merge(reviews_grouped, on='id', how='left')
train['text'] = train['text'].fillna('')
test['text'] = test['text'].fillna('')

gc.collect()

train['clean_text'] = train['text'].progress_apply(clean_text)
test['clean_text'] = test['text'].progress_apply(clean_text)

before = len(train)
train = train[train['target'] > 0].reset_index(drop=True)
after = len(train)
print(f"Отфильтровано строк без рейтинга: {before - after} (осталось {after})")

# --------------------
# Гео-фичи
# --------------------
train[['lon', 'lat']] = train['coordinates'].apply(lambda x: pd.Series(parse_coords(x)))
test[['lon', 'lat']] = test['coordinates'].apply(lambda x: pd.Series(parse_coords(x)))

both_coords = pd.concat([train[['lon', 'lat']], test[['lon', 'lat']]], axis=0).reset_index(drop=True)
mask_valid = both_coords[['lon', 'lat']].notna().all(axis=1)
n_clusters = 50
kmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=4096, n_init="auto")
kmeans.fit(both_coords.loc[mask_valid, ['lon', 'lat']])
all_labels = np.full(len(both_coords), -1, dtype=int)
all_labels[mask_valid.values] = kmeans.predict(both_coords.loc[mask_valid, ['lon', 'lat']])
train['geo_cluster'] = all_labels[:len(train)]
test['geo_cluster'] = all_labels[len(train):]

# --------------------
# Табличные фичи
# --------------------
def add_ring_features(df):
    cols_300 = [c for c in df.columns if c.endswith("_300m")]
    cols_1000 = [c for c in df.columns if c.endswith("_1000m")]
    base_common = list(set([c.replace("_300m", "") for c in cols_300]).intersection(
        [c.replace("_1000m", "") for c in cols_1000]
    ))

    for base in base_common:
        c300 = f"{base}_300m"
        c1000 = f"{base}_1000m"
        df[f"{base}_delta"] = df[c1000].astype(float) - df[c300].astype(float)
        df[f"{base}_ratio"] = safe_div(df[c300].astype(float), df[c1000].astype(float))
    return df

train = add_ring_features(train)
test = add_ring_features(test)

# --------------------
# Демографические фичи
# --------------------
def add_dem_shares(df, suffix):
    if f'female_{suffix}' in df.columns and f'male_{suffix}' in df.columns:
        total = df[f'female_{suffix}'] + df[f'male_{suffix}']
        df[f'female_share_{suffix}'] = safe_div(df[f'female_{suffix}'], total)
        df[f'male_share_{suffix}'] = safe_div(df[f'male_{suffix}'], total)
    if f'has_children_{suffix}' in df.columns and f'no_children_{suffix}' in df.columns:
        total = df[f'has_children_{suffix}'] + df[f'no_children_{suffix}']
        df[f'children_share_{suffix}'] = safe_div(df[f'has_children_{suffix}'], total)
    if f'employed_{suffix}' in df.columns and f'unemployed_{suffix}' in df.columns:
        total = df[f'employed_{suffix}'] + df[f'unemployed_{suffix}']
        df[f'employed_share_{suffix}'] = safe_div(df[f'employed_{suffix}'], total)
    if f'higher_education_{suffix}' in df.columns and f'no_higher_education_{suffix}' in df.columns:
        total = df[f'higher_education_{suffix}'] + df[f'no_higher_education_{suffix}']
        df[f'higher_edu_share_{suffix}'] = safe_div(df[f'higher_education_{suffix}'], total)

for suf in ["300m", "1000m"]:
    add_dem_shares(train, suf)
    add_dem_shares(test, suf)

# --------------------
# Текстовые фичи из каждого отзыва, усредненные по ID, + SVD
# --------------------
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = (token_embeddings * input_mask_expanded).sum(dim=1)
    sum_mask = input_mask_expanded.sum(dim=1).clamp(min=1e-9)
    return sum_embeddings / sum_mask

def embed_texts_e5_raw(texts):
    tokenizer = AutoTokenizer.from_pretrained(E5_MODEL_NAME, trust_remote_code=True)
    model = AutoModel.from_pretrained(E5_MODEL_NAME, trust_remote_code=True).to(DEVICE)
    if USE_GPU:
        model = model.half()
    model.eval()

    all_emb = []
    for i in tqdm(range(0, len(texts), E5_BATCH), desc="E5 batches"):
        batch = [f"passage: {t}" if t else "passage: ." for t in texts[i:i+E5_BATCH]]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=E5_MAX_LEN, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
            emb = mean_pooling(out, enc['attention_mask'])
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        all_emb.append(emb.detach().cpu().numpy())
    if USE_GPU:
        torch.cuda.empty_cache()
    return np.vstack(all_emb).astype(np.float32)

reviews['clean_text'] = reviews['text'].fillna('').map(clean_text)
emb_all = embed_texts_e5_raw(reviews['clean_text'].tolist())

emb_df = pd.DataFrame(emb_all)
emb_df['id'] = reviews['id'].values
e5_by_id = emb_df.groupby('id').mean().reset_index()

e5_cols_raw = [c for c in e5_by_id.columns if c != 'id']
train = train.merge(e5_by_id, on='id', how='left')
test = test.merge(e5_by_id, on='id', how='left')

e5_svd = TruncatedSVD(n_components=E5_SVD_DIM)
e5_train_svd = e5_svd.fit_transform(train[e5_cols_raw].fillna(0.0).values)
e5_test_svd = e5_svd.transform(test[e5_cols_raw].fillna(0.0).values)

e5_cols = [f"e5_{i}" for i in range(E5_SVD_DIM)]
e5_train_df = pd.DataFrame(e5_train_svd, columns=e5_cols, dtype=np.float32)
e5_test_df = pd.DataFrame(e5_test_svd, columns=e5_cols, dtype=np.float32)

gc.collect()

# --------------------
# Текстовые фичи (TF-IDF + SVD) из склеенных отзывов
# --------------------
tfidf = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    stop_words=stop_words,
    ngram_range=(1, 2),
    min_df=3, max_df=0.95
)
X_tfidf = tfidf.fit_transform(train['clean_text'])
X_tfidf_test = tfidf.transform(test['clean_text'])

svd = TruncatedSVD(n_components=TFIDF_SVD_DIM)
X_tfidf_svd = svd.fit_transform(X_tfidf)
X_test_tfidf_svd = svd.transform(X_tfidf_test)

tfidf_train_df = pd.DataFrame(X_tfidf_svd, columns=[f'svd_{i}' for i in range(TFIDF_SVD_DIM)], dtype=np.float32)
tfidf_test_df = pd.DataFrame(X_test_tfidf_svd, columns=[f'svd_{i}' for i in range(TFIDF_SVD_DIM)], dtype=np.float32)

del X_tfidf, X_tfidf_test, X_tfidf_svd, X_test_tfidf_svd
gc.collect()

# --------------------
# Собираем всё в один большой датасет
# --------------------
for df in (train, test):
    df['category'] = df['category'].fillna('unknown').astype(str)

ignore_cols = ['id', 'name', 'address', 'coordinates', 'text', 'clean_text', 'target']
num_cols = [c for c in train.columns if c not in ignore_cols and train[c].dtype != 'object']

X_num = train[num_cols].fillna(0)
X_test_num = test[num_cols].fillna(0)

X_num.columns = X_num.columns.astype(str)
X_test_num.columns = X_test_num.columns.astype(str)
scaler = StandardScaler()
X_num_scaled = pd.DataFrame(scaler.fit_transform(X_num), columns=X_num.columns, dtype=np.float32)
X_test_num_scaled = pd.DataFrame(scaler.transform(X_test_num), columns=X_test_num.columns, dtype=np.float32)

X_full = pd.concat([X_num_scaled.reset_index(drop=True), tfidf_train_df, e5_train_df], axis=1)
X_full['category'] = train['category'].values

X_test_full = pd.concat([X_test_num_scaled.reset_index(drop=True), tfidf_test_df, e5_test_df], axis=1)
X_test_full['category'] = test['category'].values

y = train['target'].astype(float).values

cat_features = [X_full.columns.get_loc('category')]

print(f"Финальные формы: X_full={X_full.shape}, X_test_full={X_test_full.shape}")

# --------------------
# Настраиваем гиперпараметры для CatBoost, XGBoost, LightGBM с помощью Optuna
# --------------------
X_full_cat = X_full.copy()
X_test_full_cat = X_test_full.copy()

X_full_num = X_full.select_dtypes(exclude=['object']).copy()
X_test_full_num = X_test_full.select_dtypes(exclude=['object']).copy()

def clean_column_names(df):
    df = df.copy()
    df.columns = [str(c).replace('[','_').replace(']','_').replace('<','_').replace('>','_') for c in df.columns]
    return df

X_full_num = clean_column_names(X_full_num)
X_test_full_num = clean_column_names(X_test_full_num)

X_tr_cat, X_val_cat, y_tr, y_val = train_test_split(X_full_cat, y, test_size=0.2, random_state=42)
X_tr_num, X_val_num, _, _ = train_test_split(X_full_num, y, test_size=0.2, random_state=42)

def objective_catboost(trial, X_train, y_train, X_valid, y_valid, cat_features):
    params = {
        "iterations": ITERATIONS,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 0.1, 1.0),
        "loss_function": "MAE",
        "eval_metric": "MAE",
        "task_type": "GPU" if USE_GPU else "CPU",
        "od_type": "Iter",
        "od_wait": 200,
        "verbose": False
    }
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train, eval_set=(X_valid, y_valid), use_best_model=True, cat_features=cat_features)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

def objective_xgb(trial, X_train, y_train, X_valid, y_valid):
    params = {
        "n_estimators": ITERATIONS,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "tree_method": "gpu_hist" if USE_GPU else "hist",
        "objective": "reg:absoluteerror",
        "eval_metric": "mae",
        "early_stopping_rounds": 200
    }
    model = XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    return mean_absolute_error(y_valid, model.predict(X_valid))

def objective_lgb(trial, X_train, y_train, X_valid, y_valid):
    params = {
        "n_estimators": ITERATIONS,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-3, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-3, 10.0, log=True),
        "objective": "mae",
        "metric": "mae",
        "verbosity": -1,
        "device_type": "gpu" if USE_GPU else "cpu"
    }
    model = LGBMRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=None)
    return mean_absolute_error(y_valid, model.predict(X_valid))

study_cat = optuna.create_study(direction="minimize", sampler=TPESampler())
study_cat.optimize(lambda t: objective_catboost(t, X_tr_cat, y_tr, X_val_cat, y_val, cat_features), n_trials=N_TRIALS)
best_params_cat = study_cat.best_params

study_xgb = optuna.create_study(direction="minimize", sampler=TPESampler())
study_xgb.optimize(lambda t: objective_xgb(t, X_tr_num, y_tr, X_val_num, y_val), n_trials=N_TRIALS)
best_params_xgb = study_xgb.best_params

study_lgb = optuna.create_study(direction="minimize", sampler=TPESampler())
study_lgb.optimize(lambda t: objective_lgb(t, X_tr_num, y_tr, X_val_num, y_val), n_trials=N_TRIALS)
best_params_lgb = study_lgb.best_params

# --------------------
# Обучаем 3 модели на 5-фолдовой кросс-валидации
# --------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_cat, oof_xgb, oof_lgb = np.zeros(len(X_full)), np.zeros(len(X_full)), np.zeros(len(X_full))
test_cat, test_xgb, test_lgb = np.zeros(len(X_test_full)), np.zeros(len(X_test_full)), np.zeros(len(X_test_full))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_full)):
    print(f"\nFOLD {fold+1}")

    X_tr_cat, X_val_cat = X_full_cat.iloc[tr_idx], X_full_cat.iloc[val_idx]
    X_tr_num, X_val_num = X_full_num.iloc[tr_idx], X_full_num.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    # ----------------------
    # CatBoost
    # ----------------------
    model_cat = CatBoostRegressor(
        iterations=ITERATIONS,
        loss_function="MAE",
        task_type="GPU" if USE_GPU else "CPU",
        od_type="Iter",
        od_wait=300,
        verbose=False,
        **best_params_cat
    )
    model_cat.fit(X_tr_cat, y_tr, eval_set=(X_val_cat, y_val), use_best_model=True, cat_features=cat_features)
    oof_cat[val_idx] = model_cat.predict(X_val_cat)
    test_cat += model_cat.predict(X_test_full_cat) / kf.n_splits

    # ----------------------
    # XGBoost
    # ----------------------
    model_xgb = XGBRegressor(
        n_estimators=ITERATIONS,
        objective="reg:absoluteerror",
        eval_metric="mae",
        tree_method="gpu_hist" if USE_GPU else "hist",
        early_stopping_rounds=300,
        **best_params_xgb
    )
    model_xgb.fit(
        X_tr_num, y_tr,
        eval_set=[(X_val_num, y_val)],
        verbose=False
    )
    oof_xgb[val_idx] = model_xgb.predict(X_val_num)
    test_xgb += model_xgb.predict(X_test_full_num) / kf.n_splits

    # ----------------------
    # LightGBM
    # ----------------------
    model_lgb = LGBMRegressor(
        n_estimators=ITERATIONS,
        objective="mae",
        metric="mae",
        verbosity=-1,
        device_type="gpu" if USE_GPU else "cpu",
        **best_params_lgb
    )
    model_lgb.fit(
        X_tr_num, y_tr,
        eval_set=[(X_val_num, y_val)],
        callbacks=None
    )
    oof_lgb[val_idx] = model_lgb.predict(X_val_num)
    test_lgb += model_lgb.predict(X_test_full_num) / kf.n_splits


# --------------------
# Взвешенный бленд
# --------------------
oof_blend = 0.5 * oof_cat + 0.3 * oof_xgb + 0.2 * oof_lgb
test_blend = 0.5 * test_cat + 0.3 * test_xgb + 0.2 * test_lgb

val_mae = mean_absolute_error(y, oof_blend)
print(f"\n✅ Final blended OOF MAE: {val_mae:.4f}")

# --------------------
# Сабмит
# --------------------
sub = pd.DataFrame({"id": test["id"], "target": np.clip(test_blend, 1.0, 5.0)})
sub['target'] = sub['target'].round(1)
sub.to_csv("submission.csv", index=False)